# 03. 피처 엔지니어링 (Feature Engineering)
## 서울 성동구 요식 가맹점 조기 경보 시스템 | 빅콘테스트 2025

> **STEP 7**: 추세 피처 계산 (3개월 차분) + 리스크 방향 정렬  
> **STEP 8**: 시간감쇠 스냅샷 집계 (λ=0.75, 05_report_tuning 최적화 결과) + 업종 내 백분위 순위

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 01_preprocessing 결과 로드 (없으면 루트의 원본 중간 산출물 사용)
import os
if os.path.exists('../outputs/panel_preprocessed.csv'):
    panel = pd.read_csv('../outputs/panel_preprocessed.csv', encoding='utf-8-sig',
                        parse_dates=['ARE_D_dt','MCT_ME_D_dt','TA_YM_dt'])
else:
    panel = pd.read_csv('../p_project_data_v2.csv', encoding='utf-8-sig',
                        parse_dates=['ARE_D_dt','MCT_ME_D_dt','TA_YM_dt'])
print(f'panel shape: {panel.shape}')


panel shape: (86246, 46)


### STEP 7-1. 시계열 정렬 & 3개월 추세 계산

In [2]:
# ================================================================
# STEP 7: 피처 엔지니어링 — 추세 & 리스크 신호 계산
# ================================================================
# panel 변수가 없을 때: 이전 스텝 결과 로드
if 'panel' not in dir():
    panel = pd.read_csv('../p_project_data_v2.csv', encoding='utf-8-sig',
                        parse_dates=['ARE_D_dt','MCT_ME_D_dt','TA_YM_dt'])

# 1. 시간 순 정렬 (groupby diff 정확성 보장)
panel = panel.sort_values(['ENCODED_MCT', 'TA_YM']).reset_index(drop=True)

# 2. 3개월 추세 계산 (store 내 diff(3))
#    양수 = 3개월 전 대비 악화, 음수 = 개선
TREND_SRC = {
    'RC_M1_SAA':                 'trend_sales',
    'RC_M1_TO_UE_CT':            'trend_trx',
    'RC_M1_AV_NP_AT':            'trend_spend',
    'M12_SME_RY_SAA_PCE_RT':     'trend_rank_ind',
    'M12_SME_BZN_SAA_PCE_RT':    'trend_rank_dist',
    'MCT_UE_CLN_REU_RAT':        'trend_return',
    'RC_M1_SHC_FLP_UE_CLN_RAT':  'trend_float',
    'M1_SME_RY_SAA_RAT':         'trend_vs_ind',
}
for src, dst in TREND_SRC.items():
    panel[dst] = panel.groupby('ENCODED_MCT')[src].transform(lambda x: x.diff(3))

### STEP 7-2. 리스크 방향 통일 (높을수록 위험)

In [3]:
# 3. 리스크 방향 통일: 높을수록 위험
# [내부 신호] 매출/거래 (bucket 6=최악 -> 높을수록 위험, 방향 OK)
panel['f_sales_lvl']   = panel['RC_M1_SAA']
panel['f_trx_lvl']     = panel['RC_M1_TO_UE_CT']
panel['f_spend_lvl']   = panel['RC_M1_AV_NP_AT']
panel['f_sales_trend'] = panel['trend_sales']     # 양수 = 버킷 악화
panel['f_trx_trend']   = panel['trend_trx']

# [내부 신호] 고객 구조
# 재방문율: 낮을수록 위험 -> 부호 반전
panel['f_return_rate']  = -panel['MCT_UE_CLN_REU_RAT']
panel['f_return_trend'] = -panel['trend_return']
panel['f_float_ratio']  = panel['RC_M1_SHC_FLP_UE_CLN_RAT']     # 유동 의존 높을수록 위험
panel['f_float_trend']  = panel['trend_float']
panel['f_resid_ratio']  = -panel['RC_M1_SHC_RSD_UE_CLN_RAT']    # 낮은 상주 고객 = 위험

# [경쟁 포지션] 높은 순위% = 낮은 순위 = 위험 (방향 OK)
panel['f_rank_ind']        = panel['M12_SME_RY_SAA_PCE_RT']
panel['f_rank_dist']       = panel['M12_SME_BZN_SAA_PCE_RT']
panel['f_rank_ind_trend']  = panel['trend_rank_ind']
panel['f_rank_dist_trend'] = panel['trend_rank_dist']
# 업종 대비 매출: 낮을수록 위험 -> 부호 반전 + 극단값 clip
panel['f_vs_ind_sales']    = -panel['M1_SME_RY_SAA_RAT'].clip(-70, 500)
panel['f_vs_ind_trend']    = -panel['trend_vs_ind']

# [외부 압력] 동종 폐업률 높을수록 외부 압력 (방향 OK)
panel['f_peer_close_ind']  = panel['M12_SME_RY_ME_MCT_RAT']
panel['f_peer_close_dist'] = panel['M12_SME_BZN_ME_MCT_RAT']

### STEP 7-3. 피처 그룹 정의 & 저장

In [4]:
# 4. 피처 그룹 정의 (이후 스텝에서 참조)
FEAT_INTERNAL    = ['f_sales_lvl','f_trx_lvl','f_spend_lvl','f_sales_trend','f_trx_trend',
                    'f_return_rate','f_return_trend','f_float_ratio','f_float_trend','f_resid_ratio']
FEAT_COMPETITIVE = ['f_rank_ind','f_rank_dist','f_rank_ind_trend','f_rank_dist_trend',
                    'f_vs_ind_sales','f_vs_ind_trend']
FEAT_EXTERNAL    = ['f_peer_close_ind','f_peer_close_dist']
FEAT_ALL         = FEAT_INTERNAL + FEAT_COMPETITIVE + FEAT_EXTERNAL

print(f'피처 수: internal={len(FEAT_INTERNAL)}, competitive={len(FEAT_COMPETITIVE)}, external={len(FEAT_EXTERNAL)}')
print(f'총 f_ 피처: {len(FEAT_ALL)}개')

# 추세 피처 NaN 비율 (첫 3개월 = NaN 정상)
trend_feats = ['f_sales_trend','f_trx_trend','f_return_trend','f_rank_ind_trend']
for col in trend_feats:
    print(f'  {col:<25}: NaN {panel[col].isna().mean()*100:.1f}%')

panel.to_csv('../p_project_features.csv', index=False, encoding='utf-8-sig')
print(f'[SAVED] p_project_features.csv  {panel.shape}')


피처 수: internal=10, competitive=6, external=2
총 f_ 피처: 18개
  f_sales_trend            : NaN 14.4%
  f_trx_trend              : NaN 14.4%
  f_return_trend           : NaN 16.4%
  f_rank_ind_trend         : NaN 14.4%
[SAVED] p_project_features.csv  (86246, 72)


---
## STEP 8. 점포별 스냅샷 — 시간감쇠 집계 + 업종 내 리스크 순위

### STEP 8-1. 감쇠 가중 평균 함수 정의

In [ ]:
# ================================================================
# STEP 8: 점포별 스냅샷 — 시간감쇠 집계 + 업종 내 리스크 순위
# ================================================================
# 설계:
#   - lambda=0.75: 최근 월 weight=1.0, 이전 월 0.75^k (k=경과 개월)
#     → 05_report_tuning.ipynb STAGE 2 그리드 서치 최적값
#   - 피어 그룹: 업종(HPSN_MCT_BZN_CD_NM) — 평균 199개 점포, 랭킹 신뢰성 확보
#   - 가중치: internal 50%, competitive 30%, external 20% (기본값, 05에서 최적화)

LAMBDA = 0.75  # 05_report_tuning.ipynb STAGE 2 최적화 결과
MIN_GROUP = 5  # 그룹 내 점포 수 < 5이면 순위 NaN 처리

def decay_wmean(vals, lam):
    v = vals[~np.isnan(vals)]
    if len(v) == 0:
        return np.nan
    w = np.array([lam ** (len(v) - 1 - i) for i in range(len(v))])
    return float(np.average(v, weights=w))

### STEP 8-2. 점포별 Decay-Weighted 집계

In [6]:
# 8-2. 점포별 집계
print(f'집계 중... (lambda={LAMBDA})')
rows = []
for mct_id, grp in panel.groupby('ENCODED_MCT', sort=False):
    grp = grp.sort_values('TA_YM')
    base = {
        'ENCODED_MCT':        mct_id,
        'MCT_NM':             grp['MCT_NM'].iat[0],
        'HPSN_MCT_ZCD_NM':    grp['HPSN_MCT_ZCD_NM'].iat[0],
        'HPSN_MCT_BZN_CD_NM': grp['HPSN_MCT_BZN_CD_NM'].iat[0],
        'HPSN_MCT_ZCD_NM_1':  grp['HPSN_MCT_ZCD_NM_1'].iat[0],
        'ARE_D_dt':           grp['ARE_D_dt'].iat[0],
        'MCT_ME_D_dt':        grp['MCT_ME_D_dt'].iat[0],
        'is_closed_obs':      int(grp['is_closed_obs'].iat[0]),
        'is_closed_all':      int(grp['is_closed_all'].iat[0]),
        'n_obs_months':       int(grp['TA_YM'].notna().sum()),
        'latest_TA_YM':       grp['TA_YM'].max(),
    }
    for feat in FEAT_ALL:
        base['dw_' + feat] = decay_wmean(grp[feat].values, LAMBDA)
    rows.append(base)

snap = pd.DataFrame(rows)
print(f'스냅샷 shape: {snap.shape}')

집계 중... (lambda=0.85)
스냅샷 shape: (4183, 29)


### STEP 8-3. 업종 내 순위 → 리스크 스코어 → 저장

In [7]:
# 8-3. 업종 내 퍼센타일 순위 (0~100, 높을수록 위험)
ind_sizes = snap.groupby('HPSN_MCT_BZN_CD_NM')['ENCODED_MCT'].transform('count')
DW_FEATS = ['dw_' + f for f in FEAT_ALL]
for dw_col in DW_FEATS:
    rank_col = 'rank_' + dw_col[3:]
    ranked = snap.groupby('HPSN_MCT_BZN_CD_NM')[dw_col].transform(
        lambda x: x.rank(pct=True, na_option='keep') * 100
    )
    snap[rank_col] = np.where(ind_sizes >= MIN_GROUP, ranked, np.nan)

# 8-4. 종합 리스크 스코어
RANK_INT  = ['rank_' + f for f in FEAT_INTERNAL]
RANK_COMP = ['rank_' + f for f in FEAT_COMPETITIVE]
RANK_EXT  = ['rank_' + f for f in FEAT_EXTERNAL]

snap['score_internal']    = snap[RANK_INT].mean(axis=1, skipna=True)
snap['score_competitive'] = snap[RANK_COMP].mean(axis=1, skipna=True)
snap['score_external']    = snap[RANK_EXT].mean(axis=1, skipna=True)
snap['risk_score'] = (
    0.5 * snap['score_internal'] +
    0.3 * snap['score_competitive'] +
    0.2 * snap['score_external']
)
snap['risk_rank_pct'] = np.where(
    ind_sizes >= MIN_GROUP,
    snap.groupby('HPSN_MCT_BZN_CD_NM')['risk_score'].transform(
        lambda x: x.rank(pct=True, na_option='keep') * 100
    ),
    np.nan
)

# 8-5. 검증
print('=' * 55)
c_score = snap[snap['is_closed_obs'] == 1]['risk_score']
o_score = snap[snap['is_closed_obs'] == 0]['risk_score']
print(f'is_closed_obs=1 평균 risk_score: {c_score.mean():.1f} (n={len(c_score)})')
print(f'is_closed_obs=0 평균 risk_score: {o_score.mean():.1f} (n={len(o_score)})')
print(f'분리 능력 (폐업 > 생존): {c_score.mean() > o_score.mean()}')

top10 = snap[snap['risk_rank_pct'] > 90]
if len(top10) > 0:
    prec = top10['is_closed_obs'].mean() * 100
    print(f'[Precision@top10%] 고위험 점포 {len(top10)}개 중 is_closed_obs=1 비율: {prec:.1f}%')

snap.to_csv('../p_project_snapshot.csv', index=False, encoding='utf-8-sig')
print(f'[SAVED] p_project_snapshot.csv  {snap.shape}')


is_closed_obs=1 평균 risk_score: 60.0 (n=30)
is_closed_obs=0 평균 risk_score: 50.4 (n=4153)
분리 능력 (폐업 > 생존): True
[Precision@top10%] 고위험 점포 316개 중 is_closed_obs=1 비율: 1.9%
[SAVED] p_project_snapshot.csv  (4183, 52)
